In [1]:
# python libraries imports
import pandas as pd
import numpy as np

# customed local functions imports
import sys
sys.path.append("./scripts")

from utils import *

#### **Dataset load and details**

In [2]:
ratings = load_ml100k()
print("============ MovieLens 100k Dataset ===========")
print(f"#Ratings: {ratings["rating"].count()} | #Users: {ratings['user'].nunique()} | #Items: {ratings['item'].nunique()} ")

============ MovieLens 100k Dataset ===========
#Ratings: 100000 | #Users: 943 | #Items: 1682 


**Effectiveness**

In [5]:
results = []

distances = ["l1", "l2"]
aggregates = ["avg", "max", "min"]

# Compute once, it doesn't depend on the variant
aa = average_ranking(ratings)

for distance in distances:
    for aggregate in aggregates:
        rep = reputation_ranking(ratings, distance, aggregate)
        tau = kendall(rep, aa)

        results.append({
            "algorithm": f"{distance.upper()}-{aggregate.upper()}",
            "kendall_tau": tau,
        })

df_clean = pd.DataFrame(results)
display(df_clean)

,algorithm,kendall_tau
0,L1-AVG,0.995186
1,L1-MAX,0.989826
2,L1-MIN,0.995939
3,L2-AVG,0.995438
4,L2-MAX,0.992436
5,L2-MIN,0.996566


**Robustness**

In [3]:
results = []

distances = ["l1", "l2"]
aggregates = ["avg", "max", "min"]
spam_ratios = [0.1, 0.3, 0.5]

# Noisy datasets: generated once per ratio, shared by all algorithms
noisy_datasets = {p: add_spammers(ratings, p) for p in spam_ratios}

# Baseline (AA): original vs noisy
aa_orig = average_ranking(ratings)
for p, noisy in noisy_datasets.items():
    results.append({
        "algorithm": "AA",
        "spam_ratio": p,
        "kendall_tau": kendall(aa_orig, average_ranking(noisy)),
    })

# Reputation-based algorithms: original vs noisy
for distance in distances:
    for aggregate in aggregates:
        rep_orig = reputation_ranking(ratings, distance, aggregate)

        for p, noisy in noisy_datasets.items():
            rep_noisy = reputation_ranking(noisy, distance, aggregate)

            results.append({
                "algorithm": f"{distance.upper()}-{aggregate.upper()}",
                "spam_ratio": p,
                "kendall_tau": kendall(rep_orig, rep_noisy),
            })

df_robust = pd.DataFrame(results)

# Table: one row per algorithm, one column per spam ratio
display(df_robust.pivot(index="algorithm", columns="spam_ratio", values="kendall_tau"))

spam_ratio,0.1,0.3,0.5
algorithm,,,
AA,0.701042,0.604888,0.580052
L1-AVG,0.702032,0.606548,0.581844
L1-MAX,0.702533,0.606175,0.580958
L1-MIN,0.701181,0.607332,0.583345
L2-AVG,0.700320,0.604330,0.579937
L2-MAX,0.700319,0.603965,0.579096
L2-MIN,0.699007,0.603642,0.579403
